# Brent's method check (Rust)

Start JupyterLab with `workshop run -- notebook`. Uses the `thole` facade crate from the working tree. After changing the Rust code, run the `:dep` cell again.

In [ ]:
:dep thole = { path = "/project/crates/thole" }

In [ ]:
use thole::core::Tolerance;
use thole::roots::{bisect, brent, Error, Options};

## 1. A first call

The Python binding runs this same code, so the bits must equal the `hex` printed by the Python notebook. `x` is an end of the final bracket.

In [ ]:
let r = brent(|x| x * x - 2.0, 0.0, 2.0, &Options::default()).unwrap();
println!("{r:?}");
println!("error = {:e}", (r.x - 2.0_f64.sqrt()).abs());
println!("bits  = {:#018x}", r.x.to_bits());

## 2. Reference roots (mpmath, 50 digits), next to bisection

In [ ]:
let opts = Options::default();
let cases: [(&str, fn(f64) -> f64, f64, f64, f64); 6] = [
    ("x^3 - x - 2", |x| x * x * x - x - 2.0, 1.0, 2.0, 1.5213797068045676),
    ("cos x - x", |x| x.cos() - x, 0.0, 1.0, 0.7390851332151607),
    ("x e^x - 1", |x| x * x.exp() - 1.0, 0.0, 1.0, 0.5671432904097838),
    ("x^2 - 2", |x| x * x - 2.0, 0.0, 2.0, std::f64::consts::SQRT_2),
    ("cos x", f64::cos, 0.0, 2.0, std::f64::consts::FRAC_PI_2),
    ("e^(20(x-.9))-1", |x| (20.0 * (x - 0.9)).exp() - 1.0, 0.0, 1.0, 0.9),
];
for (name, f, a, b, expected) in cases {
    let r = brent(f, a, b, &opts).unwrap();
    let by_bisection = bisect(f, a, b, &opts).unwrap();
    let err = (r.x - expected).abs();
    println!(
        "{name:15} x={:.16} err={err:.3e} steps={:2} (bisect: {}) ok={}",
        r.x, r.iterations, by_bisection.iterations, err <= opts.tol.bound(r.x)
    );
}

## 3. What the Lean theorems predict

- `brent_mem`, `brent_signChange`: the final bracket is inside `[a, b]` and `f` changes sign over it.
- `brent_error_of_done`: the error is at most the tolerance.
- `brent_done_of_le`: with a relative tolerance of zero, the iteration stops after at most `2 * (b - a) / xtol - 2` steps. This bound is far from sharp.

In [ ]:
fn f(x: f64) -> f64 {
    x * x - 2.0
}
let (a, b, root) = (0.0, 2.0, 2.0_f64.sqrt());
for xtol in [0.5, 0.1, 1e-2, 1e-4, 1e-8, 1e-12] {
    let opts = Options { tol: Tolerance::new(xtol, 0.0), max_iter: 100 };
    let r = brent(f, a, b, &opts).unwrap();
    let bound = (2.0 * (b - a) / xtol - 2.0).ceil().max(0.0);
    println!(
        "xtol={xtol:<6e} x={:.16} error={:.3e} bracket length={:.3e} steps={} bound={bound:e}",
        r.x, (r.x - root).abs(), r.hi - r.lo, r.iterations
    );
    assert!(a <= r.lo && r.lo <= r.hi && r.hi <= b && (r.x == r.lo || r.x == r.hi));
    assert!(f(r.lo) <= 0.0 && f(r.hi) >= 0.0);
    assert!((r.x - root).abs() <= xtol);
    assert!(r.iterations as f64 <= bound);
}

## 4. Where the interpolation does not help

A jump across zero and a root of high multiplicity: Brent's method takes more steps than bisection here. The root of multiplicity nine needs more than the default 100 steps.

In [ ]:
let many = Options { max_iter: 1000, ..Options::default() };
let hard: [(&str, fn(f64) -> f64, f64, f64); 3] = [
    ("jump at 0.7", |x| if x < 0.7 { -1.0 } else { 1e-3 }, 0.0, 1.0),
    ("(x - 0.3)^9", |x| (x - 0.3).powi(9), 0.0, 1.0),
    ("cbrt x + 0.1", |x| x.cbrt() + 0.1, -1.0, 1.0),
];
for (name, f, a, b) in hard {
    let r = brent(f, a, b, &many).unwrap();
    let by_bisection = bisect(f, a, b, &many).unwrap();
    println!("{name:13} brent: {:3} steps, x={:.15}   bisect: {} steps, x={:.15}", r.iterations, r.x, by_bisection.iterations, by_bisection.x);
}
println!("default max_iter -> {:?}", brent(|x| (x - 0.3).powi(9), 0.0, 1.0, &Options::default()));

## 5. Failures

In [ ]:
let opts = Options::default();
let few = Options { max_iter: 3, ..opts };
let zero = Options { tol: Tolerance::new(0.0, 0.0), max_iter: 100 };
let negative = Options { tol: Tolerance::new(-1.0, 0.0), max_iter: 100 };
let results: Vec<(&str, Result<_, Error>)> = vec![
    ("no sign change", brent(|x| x * x + 1.0, 0.0, 2.0, &opts)),
    ("a > b", brent(|x| x, 1.0, -1.0, &opts)),
    ("infinite endpoint", brent(|x| x, f64::NEG_INFINITY, 1.0, &opts)),
    ("negative tolerance", brent(|x| x, -1.0, 1.0, &negative)),
    ("f returns NaN", brent(|_| f64::NAN, 0.0, 1.0, &opts)),
    ("max_iter too small", brent(f64::cos, 0.0, 2.0, &few)),
    ("tolerance below spacing", brent(|x| x * x - 2.0, 0.0, 2.0, &zero)),
];
for (name, r) in &results {
    match r {
        Ok(root) => println!("{name:24} -> Ok({root:?})"),
        Err(e) => println!("{name:24} -> {e}   [{e:?}]"),
    }
}

## 6. Edge cases

A root at an endpoint is found without a step. A linear function is solved by the first secant step, and the bracket stays long because the value there is exactly zero.

In [ ]:
let edge: [(&str, fn(f64) -> f64, f64, f64); 7] = [
    ("root at a", |x| x, 0.0, 1.0),
    ("root at b", |x| x - 1.0, 0.0, 1.0),
    ("a == b on a root", |x| x - 1.0, 1.0, 1.0),
    ("linear", |x| 2.0 * x - 1.0, 0.0, 4.0),
    ("pole of 1/x", |x| 1.0 / x, -1.0, 2.0),
    ("three roots", |x| x * x * x - x, -2.0, 3.0),
    ("huge bracket", |x| x - 1.0, -f64::MAX, f64::MAX),
];
for (name, f, a, b) in edge {
    println!("{name:18} {:?}", brent(f, a, b, &Options::default()));
}